# Finite-ground CPW — HFSS Driven Terminal

## Setup and Imports

In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    FrequencySweepSpec,
    HfssDrivenSpec,
    HfssRunControl,
    LayerImport,
    LengthMeshSpec,
    ObjectBinding,
    PdkMaterial,
    TerminalPort,
    prepare_handoff,
    resolve_results,
)

import orpen_sc_pdk
from orpen_sc_pdk.materials import get_material_records
from orpen_sc_pdk.tech import LAYER

orpen_sc_pdk.activate()

## Setup and Run Controls

In [ ]:
# Choose prepare_handoff to create files, run to execute, or analyze_handoff to inspect results.
WORKFLOW_ACTION = "prepare_handoff"  # prepare_handoff | run | analyze_handoff
# Use a new unique ID for each prepared run; SCGSim refuses non-empty output directories.
RUN_ID = "cpw_finite_ground_hfss_terminal"
# Root directory for prepared geometry and handoff artifacts.
OUTPUT_ROOT = Path("notebooks/.artifacts/ComponentSimulation/CpwFiniteGround")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RETURNED_RUN_DIR = RUN_DIR  # Analysis input directory containing returned results.

## Create Simulation Component / Coupon

In [ ]:
# CPW trace length along X (um).
trace_length_um = 500.0
# Signal conductor width (um).
signal_width_um = 10.0
# Gap from signal to each ground conductor (um).
gap_um = 6.0
# Width of each ground conductor (um).
ground_width_um = 80.0
# Substrate thickness below the metal (um).
substrate_thickness_um = 500.0

coupon = gf.Component()
coupon << gf.components.rectangle(
    size=(trace_length_um, signal_width_um),
    centered=True,
    layer=LAYER.D0_TOP_M1_DRAW,
)
top_ground = coupon << gf.components.rectangle(
    size=(trace_length_um, ground_width_um), centered=True, layer=LAYER.D0_TOP_M1_DRAW
)
top_ground.movey((signal_width_um + ground_width_um) / 2 + gap_um)
bottom_ground = coupon << gf.components.rectangle(
    size=(trace_length_um, ground_width_um), centered=True, layer=LAYER.D0_TOP_M1_DRAW
)
bottom_ground.movey(-((signal_width_um + ground_width_um) / 2 + gap_um))
coupon << gf.components.rectangle(
    size=(trace_length_um, signal_width_um + 2 * (gap_um + ground_width_um)),
    centered=True,
    layer=LAYER.D0_SUBSTRATE_AREA,
)
SOURCE_GDS = OUTPUT_ROOT / "geometry" / f"{RUN_ID}.gds"
SOURCE_GDS.parent.mkdir(parents=True, exist_ok=True)
coupon.write_gds(SOURCE_GDS, with_metadata=False)
coupon.plot()

## Initialize AEDT Project / App

In [ ]:
# AEDT version used for project generation.
aedt_version = "2024.2"
# AEDT project name.
project_name = RUN_ID
# AEDT design name.
design_name = "CpwFiniteGroundTerminal"

## Import GDS and Build the HFSS/Q3D/Q2D Model

In [ ]:
# GDS layer number/datatype, AEDT name, bottom z and thickness (um).
layer_imports = (
    LayerImport(1, 0, "D0_TOP_M1", 0.0, 0.0),
    LayerImport(201, 0, "D0_SUBSTRATE", -substrate_thickness_um, 0.0),
)
# Imported object name, source layer, semantic role, and material ID.
object_bindings = (
    ObjectBinding("D0_TOP_M1_1", 1, "signal", "Nb"),
    ObjectBinding("D0_TOP_M1_2", 1, "ground", "Nb"),
    ObjectBinding("D0_TOP_M1_3", 1, "ground", "Nb"),
    ObjectBinding("D0_SUBSTRATE_4", 201, "substrate", "Si"),
)

## Geometry Verification

In [ ]:
display(coupon)

## Materials and Boundaries

In [ ]:
material_records = get_material_records()
materials = {
    material_id: PdkMaterial(
        material_id,
        material_records[material_id]["material_kind"],
        material_records[material_id]["is_superconducting"],
        material_records[material_id]["aedt_library_name"],
    )
    for material_id in ("vacuum", "Si", "Nb")
}
# Padding around the model in -X, +X, -Y, +Y, -Z, +Z directions (um).
region_padding_um = (0.0, 0.0, 100.0, 100.0, 1000.0, 1000.0)

## Ports / Nets / Excitations

In [ ]:
# Ground object names used by terminal reference and mesh.
ground_objects = ("D0_TOP_M1_2", "D0_TOP_M1_3")
ports = (
    # Port ID, name, direction, and associated ground objects.
    TerminalPort(1, "o1", "-X", ground_objects),
    TerminalPort(2, "o2", "+X", ground_objects),
)

## Simulation Setup

In [ ]:
# Sweep lower frequency (GHz).
frequency_start_ghz = 3.0
# Sweep upper frequency (GHz).
frequency_stop_ghz = 8.0
# Number of frequency samples.
frequency_points = 20_000
spec = HfssDrivenSpec(
    mode="terminal",
    gds_path=SOURCE_GDS,
    project_name=project_name,
    design_name=design_name,
    materials=materials,
    vacuum_material_id="vacuum",
    layer_imports=layer_imports,
    object_bindings=object_bindings,
    ports=ports,
    run_control=HfssRunControl(
        "Setup1",
        "Fast20000",
        FrequencySweepSpec(frequency_start_ghz, frequency_stop_ghz, frequency_points),
    ),
    region_padding_um=region_padding_um,
    length_mesh=LengthMeshSpec(("D0_TOP_M1_1",), ground_objects, signal_width_um),
    aedt_version=aedt_version,
)

## Simulation Configuration

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = prepare_handoff(spec=spec, output_dir=RUN_DIR)
display(HANDOFF)

## Solve and Export

In [ ]:
if WORKFLOW_ACTION == "run":
    subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
RESULT = resolve_results(RETURNED_RUN_DIR) if WORKFLOW_ACTION == "analyze_handoff" else None
display(RESULT)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if RESULT is not None:
    display(RESULT.physics_results())

### Simulation Performance / Benchmarks

In [ ]:
if RESULT is not None:
    display(RESULT.simulation_benchmark())

## Save and Release AEDT

In [ ]:
display(RESULT.project_path if RESULT is not None else HANDOFF.archive_path)